# Task 41: Deep Learning vs Traditional Machine Learning

Compare manually implemented classical models and a scratch neural network using train/test metrics and generalization gaps. Discuss interpretability, data needs, complexity, and practical limitations.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

candidates = [Path('heart.csv'), Path('Task_41_Deep_Learning_vs_Traditional_ML/heart.csv'), Path('Heart_Disease_Tasks/Task_41_Deep_Learning_vs_Traditional_ML/heart.csv')]
data = pd.read_csv(next(p for p in candidates if p.is_file())).drop_duplicates().reset_index(drop=True)
features = [c for c in ['age','sex','cp','trestbps','chol','fbs','restecg','thalach','exang','oldpeak','slope','ca','thal'] if c in data]
X, y = data[features].to_numpy(float), data.target.to_numpy(int)
rng = np.random.default_rng(41); train_idx=[]; test_idx=[]
for label in np.unique(y):
    ids=np.flatnonzero(y==label); rng.shuffle(ids); cut=round(.75*len(ids)); train_idx.extend(ids[:cut]); test_idx.extend(ids[cut:])
train_idx,test_idx=np.array(train_idx),np.array(test_idx); rng.shuffle(train_idx); rng.shuffle(test_idx)
X_train,X_test,y_train,y_test=X[train_idx],X[test_idx],y[train_idx],y[test_idx]
mean,scale=X_train.mean(0),X_train.std(0); scale[scale==0]=1
X_train,X_test=(X_train-mean)/scale,(X_test-mean)/scale


def sigmoid(z): return 1/(1+np.exp(-np.clip(z,-35,35)))

class ManualLogistic:
    def __init__(self, l2=0.02, rate=0.06, steps=1400): self.l2,self.rate,self.steps=l2,rate,steps
    def fit(self,x,y):
        self.w=np.zeros(x.shape[1]); self.b=0.
        for _ in range(self.steps):
            p=sigmoid(x@self.w+self.b); error=p-y
            self.w-=self.rate*(x.T@error/len(y)+self.l2*self.w); self.b-=self.rate*error.mean()
        return self
    def predict(self,x): return (sigmoid(x@self.w+self.b)>=.5).astype(int)

class ManualTree:
    def __init__(self,max_depth=4): self.max_depth=max_depth
    def fit(self,x,y): self.root=self._grow(x,y,0); return self
    def _grow(self,x,y,depth):
        counts=np.bincount(y,minlength=2); node={'pred':int(np.argmax(counts))}
        if depth>=self.max_depth or len(np.unique(y))<2 or len(y)<8: return node
        parent=1-np.sum((counts/len(y))**2); best=None
        for col in range(x.shape[1]):
            values=np.unique(x[:,col])
            for threshold in (values[:-1]+values[1:])/2:
                mask=x[:,col]<=threshold
                if not mask.any() or mask.all(): continue
                left=np.bincount(y[mask],minlength=2)/mask.sum(); right=np.bincount(y[~mask],minlength=2)/(~mask).sum()
                gain=parent-mask.mean()*(1-np.sum(left**2))-(~mask).mean()*(1-np.sum(right**2))
                if best is None or gain>best[0]: best=(gain,col,threshold,mask)
        if best is None or best[0]<=0:return node
        _,col,threshold,mask=best
        node.update(col=col,threshold=threshold,left=self._grow(x[mask],y[mask],depth+1),right=self._grow(x[~mask],y[~mask],depth+1))
        return node
    def predict(self,x):
        result=[]
        for row in x:
            node=self.root
            while 'col' in node: node=node['left'] if row[node['col']]<=node['threshold'] else node['right']
            result.append(node['pred'])
        return np.array(result)

class ManualNeuralNet:
    def __init__(self,hidden=(16,8),seed=41): self.hidden,self.seed=hidden,seed
    def fit(self,x,y,epochs=100,rate=.01,batch=32):
        random=np.random.default_rng(self.seed); dims=(x.shape[1],*self.hidden,1)
        self.w=[random.normal(0,np.sqrt(2/dims[i]),(dims[i],dims[i+1])) for i in range(len(dims)-1)]
        self.b=[np.zeros((1,dims[i+1])) for i in range(len(dims)-1)]; self.loss=[]; target=y[:,None]
        for _ in range(epochs):
            order=random.permutation(len(x)); total=0.
            for start in range(0,len(x),batch):
                ids=order[start:start+batch]; activ=[x[ids]]
                for layer in range(len(self.hidden)): activ.append(np.maximum(activ[-1]@self.w[layer]+self.b[layer],0))
                p=sigmoid(activ[-1]@self.w[-1]+self.b[-1]); yy=target[ids]
                total+=float(-np.sum(yy*np.log(p+1e-12)+(1-yy)*np.log(1-p+1e-12)))
                grad=(p-yy)/len(ids); dw=[None]*len(self.w); db=[None]*len(self.b)
                dw[-1]=activ[-1].T@grad; db[-1]=grad.sum(0,keepdims=True); grad=grad@self.w[-1].T
                for layer in range(len(self.hidden)-1,-1,-1):
                    grad*=activ[layer+1]>0; dw[layer]=activ[layer].T@grad; db[layer]=grad.sum(0,keepdims=True); grad=grad@self.w[layer].T
                for layer in range(len(self.w)): self.w[layer]-=rate*dw[layer]; self.b[layer]-=rate*db[layer]
            self.loss.append(total/len(x))
        return self
    def predict(self,x):
        a=x
        for layer in range(len(self.hidden)): a=np.maximum(a@self.w[layer]+self.b[layer],0)
        return (sigmoid(a@self.w[-1]+self.b[-1]).ravel()>=.5).astype(int)


def manual_metrics(actual,predicted):
    tp=np.sum((actual==1)&(predicted==1)); tn=np.sum((actual==0)&(predicted==0)); fp=np.sum((actual==0)&(predicted==1)); fn=np.sum((actual==1)&(predicted==0))
    precision=tp/(tp+fp) if tp+fp else 0; recall=tp/(tp+fn) if tp+fn else 0
    return {'accuracy':(tp+tn)/len(actual),'precision':precision,'recall':recall,'f1':2*precision*recall/(precision+recall) if precision+recall else 0}

models={'Logistic regression':ManualLogistic().fit(X_train,y_train),'Decision tree':ManualTree(4).fit(X_train,y_train),'Neural network':ManualNeuralNet().fit(X_train,y_train)}
rows=[]
for name,model in models.items():
    train=manual_metrics(y_train,model.predict(X_train)); test=manual_metrics(y_test,model.predict(X_test))
    rows.append({'Model':name,'Train accuracy':train['accuracy'],'Test accuracy':test['accuracy'],'Accuracy gap':train['accuracy']-test['accuracy'],'Test precision':test['precision'],'Test recall':test['recall'],'Test F1':test['f1']})
results=pd.DataFrame(rows).set_index('Model')
print('Unique records:',len(data)); display(results.round(3))
fig,ax=plt.subplots(figsize=(9,4)); positions=np.arange(len(results))
ax.bar(positions-.18,results['Train accuracy'],width=.36,label='Train'); ax.bar(positions+.18,results['Test accuracy'],width=.36,label='Test')
ax.set_xticks(positions,results.index,rotation=12); ax.set_ylabel('Accuracy'); ax.set_title('Train/test performance and generalization gap'); ax.legend(); ax.grid(axis='y',alpha=.2); plt.show()
print('Classical methods are typically data-efficient, fast on small tabular data, and easier to interpret. Neural networks learn nonlinear interactions but need careful tuning, more data, and stronger validation; larger training accuracy gaps indicate possible overfitting. Clinical decisions require external validation and expert oversight.')